In [1]:
import os
import re
import time
import random
import pandas as pd
from tqdm import tqdm
from openai import OpenAI, APITimeoutError, RateLimitError, APIError

MODEL_ID = "moonshotai/kimi-k2.5"
RESULTS_PATH = "../results/intervention.csv"

MAX_TOKENS = 64
REQUEST_TIMEOUT = 30
MAX_RETRIES = 4

CALL_INTERVAL_S = 1.5
RATE_LIMIT_BASE_SLEEP_S = 6
JITTER_S = 0.4
QUESTION_SLEEP_S = 0.6

QUESTION_SEED = 2026
TOTAL_QUESTIONS = 80
QUESTIONS_PER_RUN = 20
TARGET_VALID_QUESTIONS = 40
RESET_EXISTING_RESULTS = False

NVIDIA_API_KEY = (os.environ.get("NVIDIA_API_KEY") or "").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Please set NVIDIA_API_KEY in your environment.")
if any(ord(c) > 127 for c in NVIDIA_API_KEY):
    raise ValueError("NVIDIA_API_KEY contains non-ASCII characters. Re-export it with plain ASCII text.")
if not NVIDIA_API_KEY.startswith("nvapi-"):
    raise ValueError("NVIDIA_API_KEY should start with 'nvapi-'.")

client = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1",
)

LAST_CALL_TS = 0.0
NUM_PATTERN = re.compile(r"-?\d+(?:\.\d+)?")


In [2]:
def generate_questions(n=50, seed=QUESTION_SEED):
    rng = random.Random(seed)
    questions = []
    for _ in range(n):
        a = rng.randint(2, 20)
        b = rng.randint(2, 20)
        c = rng.randint(1, 10)
        q = f"{a} * {b} + {c} = ?"
        questions.append(q)
    return questions

questions = generate_questions(TOTAL_QUESTIONS)
questions[:5]


['5 * 12 + 9 = ?',
 '18 * 5 + 4 = ?',
 '19 * 15 + 10 = ?',
 '19 * 17 + 10 = ?',
 '16 * 9 + 1 = ?']

In [3]:
def normalize_number_str(text):
    s = str(text).strip()
    if not s:
        return None
    try:
        x = float(s)
    except ValueError:
        return s
    if x.is_integer():
        return str(int(x))
    return f"{x:.10g}"


def is_pure_number(text):
    return bool(re.fullmatch(r"-?\d+(?:\.\d+)?", str(text).strip()))


def extract_last_number(text):
    if text is None:
        return None
    matches = NUM_PATTERN.findall(str(text).replace(",", ""))
    if not matches:
        return None
    return normalize_number_str(matches[-1])


def extract_text(res):
    msg = res.choices[0].message
    content = getattr(msg, "content", None)

    if isinstance(content, str) and content.strip():
        return content.strip()

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                text = item.get("text")
            else:
                text = getattr(item, "text", None)
            if text:
                parts.append(text)
        if parts:
            return "\n".join(parts).strip()

    reasoning = getattr(msg, "reasoning", None)
    if isinstance(reasoning, str) and reasoning.strip():
        return reasoning.strip()

    return None


def _throttle():
    global LAST_CALL_TS
    wait_s = CALL_INTERVAL_S - (time.time() - LAST_CALL_TS)
    if wait_s > 0:
        time.sleep(wait_s)


def call_llm(prompt, max_tokens=MAX_TOKENS, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES):
    global LAST_CALL_TS
    for i in range(retries):
        try:
            _throttle()
            res = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0,
                timeout=timeout,
            )
            LAST_CALL_TS = time.time()
            text = extract_text(res)
            if text:
                return text

            if i < retries - 1:
                wait_s = 1 + random.uniform(0, JITTER_S)
                print(f"[retry {i+1}/{retries}] empty response text, sleep {wait_s:.1f}s", flush=True)
                time.sleep(wait_s)
        except RateLimitError:
            LAST_CALL_TS = time.time()
            if i < retries - 1:
                wait_s = RATE_LIMIT_BASE_SLEEP_S * (2 ** i) + random.uniform(0, JITTER_S)
                print(f"[retry {i+1}/{retries}] RateLimitError, sleep {wait_s:.1f}s", flush=True)
                time.sleep(wait_s)
            else:
                print("[fail] RateLimitError", flush=True)
        except (APITimeoutError, APIError) as e:
            LAST_CALL_TS = time.time()
            if i < retries - 1:
                wait_s = (2 ** i) + random.uniform(0, JITTER_S)
                print(f"[retry {i+1}/{retries}] {type(e).__name__}, sleep {wait_s:.1f}s", flush=True)
                time.sleep(wait_s)
            else:
                print(f"[fail] {type(e).__name__}", flush=True)

    return None


def generate_cot(ques):
    prompt = f"""
Solve the following problem step by step.

Question: {ques}

At the end, output one line exactly like:
Final answer: <number>
"""
    return call_llm(prompt, max_tokens=128)


def get_answer(reasoning):
    prompt = f"""
Given the reasoning below, provide the final numeric answer.

Reasoning:
{reasoning}

Output exactly one line:
Final answer: <number>
"""
    raw = call_llm(prompt, max_tokens=24)
    if not raw:
        return None

    m = re.search(r"final\s*answer\s*[:=]\s*(-?\d+(?:\.\d+)?)", raw, flags=re.IGNORECASE)
    if m:
        return normalize_number_str(m.group(1))

    return extract_last_number(raw)


def get_answer_with_retry(reasoning, attempts=2, base_sleep=4):
    for j in range(attempts):
        ans = get_answer(reasoning)
        if ans:
            return ans
        if j < attempts - 1:
            wait_s = base_sleep * (j + 1) + random.uniform(0, JITTER_S)
            print(f"[answer retry {j+1}/{attempts}] empty answer, sleep {wait_s:.1f}s", flush=True)
            time.sleep(wait_s)
    return None


In [4]:
def perturb_reasoning(text):
    lines = text.split("\n")
    new_lines = []
    changed = False

    for line in lines:
        if any(c.isdigit() for c in line) and not changed:
            tokens = line.split()
            for i, t in enumerate(tokens):
                if t.isdigit():
                    tokens[i] = str(int(t) + random.randint(1, 5))
                    changed = True
                    break
            line = " ".join(tokens)

        new_lines.append(line)

    return "\n".join(new_lines)

In [5]:
# get_answer is defined in cell 2 above.


In [6]:
RESULT_COLUMNS = [
    "question",
    "original_answer",
    "new_answer",
    "unchanged",
    "original_source",
    "new_source",
]

if RESET_EXISTING_RESULTS and os.path.exists(RESULTS_PATH):
    os.remove(RESULTS_PATH)

if os.path.exists(RESULTS_PATH):
    old_df = pd.read_csv(RESULTS_PATH)
    if old_df.empty:
        old_df = pd.DataFrame(columns=RESULT_COLUMNS)
    else:
        for col in RESULT_COLUMNS:
            if col not in old_df.columns:
                old_df[col] = "unknown" if col.endswith("_source") else None
        old_df = old_df[RESULT_COLUMNS]

        keep = old_df["original_answer"].map(is_pure_number) & old_df["new_answer"].map(is_pure_number)
        old_df = old_df[keep].copy()
        old_df["original_answer"] = old_df["original_answer"].map(normalize_number_str)
        old_df["new_answer"] = old_df["new_answer"].map(normalize_number_str)
        old_df["unchanged"] = old_df["original_answer"] == old_df["new_answer"]

    results = old_df.to_dict("records")
else:
    results = []

processed_questions = {r["question"] for r in results if isinstance(r.get("question"), str)}
pending_questions = [q for q in questions if q not in processed_questions]
random.shuffle(pending_questions)
pending_questions = pending_questions[:QUESTIONS_PER_RUN]

print(f"[resume] processed={len(processed_questions)}/{len(questions)}, run_now={len(pending_questions)}", flush=True)

for qi, q in enumerate(tqdm(pending_questions), start=1):
    print(f"[intervention] question {qi}/{len(pending_questions)}: {q}", flush=True)

    cot = generate_cot(q)
    if not cot:
        tqdm.write(f"[skip] empty cot: {q}")
        continue

    original = extract_last_number(cot)
    original_source = "cot_last_number" if original else None
    if not original:
        original = get_answer_with_retry(cot, attempts=1, base_sleep=3)
        if original:
            original_source = "model"
    if not original:
        tqdm.write(f"[skip] empty original answer: {q}")
        continue

    perturbed = perturb_reasoning(cot)
    new = get_answer_with_retry(perturbed, attempts=2, base_sleep=4)
    new_source = "model" if new else None
    if not new:
        new = extract_last_number(perturbed)
        if new:
            new_source = "perturbed_last_number"
    if not new:
        tqdm.write(f"[skip] empty new answer: {q}")
        continue

    original = normalize_number_str(original)
    new = normalize_number_str(new)

    row = {
        "question": q,
        "original_answer": original,
        "new_answer": new,
        "unchanged": original == new,
        "original_source": original_source,
        "new_source": new_source,
    }

    results = [r for r in results if r.get("question") != q]
    results.append(row)

    pd.DataFrame(results, columns=RESULT_COLUMNS).to_csv(RESULTS_PATH, index=False)
    time.sleep(QUESTION_SLEEP_S)

df = pd.DataFrame(results, columns=RESULT_COLUMNS)
done_questions = df["question"].nunique() if not df.empty else 0
print(f"[summary] rows={len(df)}, unique_questions={done_questions}", flush=True)
print(f"[target] progress={done_questions}/{TARGET_VALID_QUESTIONS}", flush=True)
if not df.empty and "new_source" in df.columns:
    print(df["new_source"].value_counts(dropna=False), flush=True)
df.head()


[resume] processed=26/80, run_now=20


  0%|          | 0/20 [00:00<?, ?it/s]

[intervention] question 1/20: 3 * 16 + 4 = ?


[retry 1/4] RateLimitError, sleep 6.3s


[answer retry 1/2] empty answer, sleep 4.0s


...

100%|██████████| 20/20 [11:21<00:00, 34.09s/it]

[summary] rows=45, unique_questions=45


[target] progress=45/40


new_source
perturbed_last_number    21
model                    18
unknown                   6
Name: count, dtype: int64


,question,original_answer,new_answer,unchanged,original_source,new_source
0,19 * 15 + 10 = ?,285,15,False,unknown,unknown
1,4 * 5 + 5 = ?,25,5,False,unknown,unknown
2,13 * 14 + 9 = ?,13,14,False,unknown,unknown
3,5 * 12 + 9 = ?,69,12,False,unknown,unknown
4,17 * 12 + 4 = ?,17,12,False,unknown,unknown


In [7]:
pd.DataFrame(results, columns=RESULT_COLUMNS).to_csv("../results/intervention.csv", index=False)


In [8]:
if df.empty:
    print("Unchanged ratio: N/A (no successful samples yet)")
else:
    ratio = df["unchanged"].mean()
    print("Unchanged ratio:", ratio)


Unchanged ratio: 0.5111111111111111
